# Import Dependenices

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# Dataset Load

In [ ]:
train_df = pd.read_csv("datasets/CICIOMT/train/ciciomt2024_train_balanced.csv")
val_df = pd.read_csv("datasets/CICIOMT/val/ciciomt2024_validation_balanced.csv")
test_df = pd.read_csv("datasets/CICIOMT/test/ciciomt2024_test_balanced.csv")

# Define feature and target columns

In [ ]:
non_feature_cols = [
    "source_file",
    "attack_type",
    "binary_label"
]

feature_cols = [col for col in train_df.columns if col not in non_feature_cols]
target_label = "binary_label"

# Train, Validation and Test

In [ ]:
# Creating features for training
X_train = train_df[feature_cols]
y_train = train_df[target_label]

# Creating features for validation
X_val = val_df[feature_cols]
y_val = val_df[target_label]

# Creating features for test
X_test = test_df[feature_cols]
y_test = test_df[target_label]

# SHAP Importance on Full XGBoost

In [ ]:
# XGBoost model
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

# train
xgb.fit(X_train, y_train)

# Get feature importance from trained XGBoost model
importance_values = xgb.feature_importances_

# Create Dataframe
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importance_values
})

# Sort by importance
importance_df = importance_df.sort_values(by="importance", ascending=False)

X_shap = X_test.sample(n=5000, random_state=42)
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)
shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    by="mean_abs_shap",
    ascending=False
)

# Behavioral Feature Group Mapping

In [ ]:
feature_groups = {
    "Header_Length": "Packet-size behavior",
    "Protocol Type": "Protocol behavior",
    "Duration": "Timing/rate behavior",
    "Rate": "Timing/rate behavior",
    "Srate": "Timing/rate behavior",
    "Drate": "Timing/rate behavior",

    "fin_flag_number": "TCP flag/count behavior",
    "syn_flag_number": "TCP flag/count behavior",
    "rst_flag_number": "TCP flag/count behavior",
    "psh_flag_number": "TCP flag/count behavior",
    "ack_flag_number": "TCP flag/count behavior",
    "ece_flag_number": "TCP flag/count behavior",
    "cwr_flag_number": "TCP flag/count behavior",
    "ack_count": "TCP flag/count behavior",
    "syn_count": "TCP flag/count behavior",
    "fin_count": "TCP flag/count behavior",
    "rst_count": "TCP flag/count behavior",

    "HTTP": "Protocol behavior",
    "HTTPS": "Protocol behavior",
    "DNS": "Protocol behavior",
    "Telnet": "Protocol behavior",
    "SMTP": "Protocol behavior",
    "SSH": "Protocol behavior",
    "IRC": "Protocol behavior",
    "TCP": "Protocol behavior",
    "UDP": "Protocol behavior",
    "DHCP": "Protocol behavior",
    "ARP": "Protocol behavior",
    "ICMP": "Protocol behavior",
    "IGMP": "Protocol behavior",
    "IPv": "Protocol behavior",
    "LLC": "Protocol behavior",

    "Tot sum": "Packet-size behavior",
    "Min": "Packet-size behavior",
    "Max": "Packet-size behavior",
    "AVG": "Packet-size behavior",
    "Std": "Packet-size behavior",
    "Tot size": "Packet-size behavior",

    "IAT": "Timing/rate behavior",
    "Number": "Statistical flow behavior",
    "Magnitue": "Statistical flow behavior",
    "Radius": "Statistical flow behavior",
    "Covariance": "Statistical flow behavior",
    "Variance": "Statistical flow behavior",
    "Weight": "Statistical flow behavior"
}

# Apply the behavioral features

In [ ]:
shap_importance["behavior_group"] = shap_importance["feature"].map(feature_groups)
print("Missing group features:")
print(shap_importance[shap_importance["behavior_group"].isna()])

shap_importance.head()

# Behavioral Group Contribution

In [ ]:
group_contribution = shap_importance.groupby("behavior_group")["mean_abs_shap"].sum().reset_index()

total_shap = group_contribution["mean_abs_shap"].sum()

group_contribution["percentage"] = (
    group_contribution["mean_abs_shap"] / total_shap * 100
)

group_contribution = group_contribution.sort_values(
    by="percentage",
    ascending=False
)

print(group_contribution)

plt.figure(figsize=(9, 5))
plt.barh(group_contribution["behavior_group"], group_contribution["percentage"])
plt.gca().invert_yaxis()
plt.xlabel("SHAP Contribution (%)")
plt.ylabel("Behavior Group")
plt.title("Behavior-Group SHAP Contribution: XGBoost Full 45 Features")
plt.tight_layout()
plt.show()